# 01_load_transform_clean

Loads raw BUDDI exports from `data/raw/` and writes flattened CSVs to:

- `data/processed/struct/`
- `data/processed/sem/`

Steps to follow:
1. Discover raw files
2. Load raw CSV(s)
3. `raw_to_struct_*`
4. `struct_to_sem_*`
5. Write `struct_*.csv` to `data/processed/struct/`
6. Write `sem_*.csv` to `data/processed/sem/`

Each datasource cell produces both `struct_*.csv` and `sem_*.csv`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import shutil


REPO_ROOT = Path("..")

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT


PosixPath('/Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project')

In [16]:
from pathlib import Path

RAW_DIR = REPO_ROOT / "data" / "input"
OUT_STRUCT_DIR = REPO_ROOT / "data" / "processed" / "struct"
OUT_SEM_DIR = REPO_ROOT / "data" / "processed" / "sem"
OUT_RAW_DIR = REPO_ROOT / "data" / "processed" / "raw"

OUT_STRUCT_DIR.mkdir(parents=True, exist_ok=True)
OUT_SEM_DIR.mkdir(parents=True, exist_ok=True)
OUT_RAW_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, OUT_STRUCT_DIR, OUT_SEM_DIR, OUT_RAW_DIR
DELIMITER = ";"
DATE_COLUMN_SUBSTRING="date"
DAYFIRST = True

In [17]:
from jbhi_eval.io import load_csv_with_date_parsing

PATIENT_GLOB = "*BUDDI_export*.csv"
PROM_GLOB = "*BUDDI_PROM*.csv"

def copy_raw_to_processed(src_path: Path, dst_filename: str) -> Path:
    """Copy raw input CSV byte-for-byte into `data/processed/raw` with a standardized name."""
    OUT_RAW_DIR.mkdir(parents=True, exist_ok=True)
    dst_path = OUT_RAW_DIR / dst_filename
    shutil.copy2(src_path, dst_path)
    return dst_path



## Patients

Raw → structural → semantic (Option A: semantic == structural)

In [35]:
from jbhi_eval.transform_patient import (
    raw_to_struct_patient,
    struct_to_sem_patient,
    STRUCT_OUT_NAME as PATIENT_STRUCT_NAME,
    SEM_OUT_NAME as PATIENT_SEM_NAME,
)

patient_files = sorted(RAW_DIR.glob(PATIENT_GLOB))
if not patient_files:
    raise FileNotFoundError(f"No patient files found in {RAW_DIR} matching {PATIENT_GLOB}")

patient_path = patient_files[0]
print("Patient input:", patient_path)

df_patient_raw = load_csv_with_date_parsing(
    patient_path,
    delimiter=";",
    date_column_substring="date",
    dayfirst=True,
    date_errors="coerce",  
)

df_patient_struct = raw_to_struct_patient(df_patient_raw)
df_patient_sem = struct_to_sem_patient(df_patient_struct, mapping_df=None)

patient_struct_path = OUT_STRUCT_DIR / PATIENT_STRUCT_NAME
patient_sem_path = OUT_SEM_DIR / PATIENT_SEM_NAME

patient_raw_copy = copy_raw_to_processed(patient_path, PATIENT_STRUCT_NAME.replace("struct_", "raw_", 1))
print("Raw copy ->", patient_raw_copy)

df_patient_struct.to_csv(patient_struct_path, index=False)
df_patient_sem.to_csv(patient_sem_path, index=False)

print("Wrote:", patient_struct_path)
print("Wrote:", patient_sem_path)


Patient input: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/input/BUDDI_export_20250317.csv
Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_patients.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_patients.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_patients.csv


## PROMs (QuestionnaireResponse)

Raw → structural → semantic

In [24]:
from jbhi_eval.transform_prom import (
    extract_prom_type,
    raw_to_struct_prom,
    struct_to_sem_prom,
    STRUCT_OUT_TEMPLATE,
    SEM_OUT_TEMPLATE,
)

prom_files = sorted(RAW_DIR.glob(PROM_GLOB))
if not prom_files:
    print(f"No PROM files found in {RAW_DIR} matching {PROM_GLOB}")
else:
    print(f"Found {len(prom_files)} PROM file(s).")

for prom_path in prom_files:
    prom_type = extract_prom_type(prom_path)
    print("\nPROM input:", prom_path)
    print("PROM type:", prom_type)

    df_prom_raw = load_csv_with_date_parsing(
        prom_path,
        delimiter=";",
        date_column_substring="date",
        dayfirst=True,
        date_errors="raise",  
    )

    df_prom_struct = raw_to_struct_prom(df_prom_raw, prom_type=prom_type)
    df_prom_sem = struct_to_sem_prom(df_prom_struct, mapping_df=None)

    out_struct = OUT_STRUCT_DIR / STRUCT_OUT_TEMPLATE.format(prom_type=prom_type)
    out_sem = OUT_SEM_DIR / SEM_OUT_TEMPLATE.format(prom_type=prom_type)

    raw_out_name = out_struct.name.replace("struct_", "raw_", 1)
    raw_copy_path = copy_raw_to_processed(prom_path, raw_out_name)
    print("Raw copy ->", raw_copy_path)

    df_prom_struct.to_csv(out_struct, index=False)
    df_prom_sem.to_csv(out_sem, index=False)

    print("Wrote:", out_struct)
    print("Wrote:", out_sem)


Found 4 PROM file(s).

PROM input: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/input/BUDDI_PROM_angst_export_20250317.csv
PROM type: angst
Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_PROM_angst_export.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_PROM_angst_export.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_PROM_angst_export.csv

PROM input: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/input/BUDDI_PROM_cognitief_functioneren_export_20250317.csv
PROM type: cognitief_functioneren
Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_PROM_cognitief_functioneren_export.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_PROM_cognitief_functioneren_export.csv
Wrote: /

## Vineland
Raw → structural → semantic



In [25]:
# --- Vineland (Observation) ---
from jbhi_eval.transform_vineland import (
    raw_to_struct_vineland,
    struct_to_sem_vineland,
    STRUCT_OUT_NAME,
    SEM_OUT_NAME,
)

vineland_files = sorted(RAW_DIR.glob("*BUDDI_Vineland_ex*.csv"))
if not vineland_files:
    print("No raw Vineland CSV found matching '*BUDDI_Vineland_ex*.csv' in data/raw/")
else:
    if len(vineland_files) > 1:
        print(f"Multiple Vineland files found. Using the first one: {vineland_files[0].name}")

    vineland_path = vineland_files[0]

    df_vl_raw = load_csv_with_date_parsing(
        vineland_path,
        delimiter=";",
        date_column_substring="date",  
        dayfirst=True,
        date_errors="raise", 
    )

    df_vl_struct = raw_to_struct_vineland(df_vl_raw)
    df_vl_sem = struct_to_sem_vineland(df_vl_struct)

    vineland_raw_copy = copy_raw_to_processed(vineland_path, STRUCT_OUT_NAME.replace("struct_", "raw_", 1))
    print("Raw copy ->", vineland_raw_copy)


    df_vl_struct.to_csv(OUT_STRUCT_DIR / STRUCT_OUT_NAME, index=False)
    df_vl_sem.to_csv(OUT_SEM_DIR / SEM_OUT_NAME, index=False)

    print("Vineland STRUCT+SEM outputs written.")
    print("Wrote:", OUT_STRUCT_DIR / STRUCT_OUT_NAME)
    print("Wrote:", OUT_SEM_DIR / SEM_OUT_NAME)


/Users/feline.mollerus/Library/Python/3.9/lib/python/site-packages/fhir_core/fhirabstractmodel.py:567: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `CodeableConcept` - serialized value may not be as expected [input_value={'system': 'http://buddi....aptive behavior scales'}, input_type=dict])
  return serialize(value)
/Users/feline.mollerus/Library/Python/3.9/lib/python/site-packages/fhir_core/fhirabstractmodel.py:567: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `CodeableConcept` - serialized value may not be as expected [input_value={'system': 'http://vinela...y': 'com', 'text': 'RT'}, input_type=dict])
  return serialize(value)
/Users/feline.mollerus/Library/Python/3.9/lib/python/site-packages/fhir_core/fhirabstractmodel.py:567: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `CodeableConcept` - serialized value may not be as expected [input_value={'s

Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_Vineland.csv
Vineland STRUCT+SEM outputs written.
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_Vineland.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_Vineland.csv


## Perinatal History
Raw → structural → semantic


In [26]:
# -------- Perinatal history (STRUCT + SEM) --------
from jbhi_eval.io import load_csv_with_date_parsing
from jbhi_eval.transform_perinatal import (
    raw_to_struct_perinatal,
    struct_to_sem_perinatal,
    STRUCT_OUT_NAME,
    SEM_OUT_NAME,
)

export_files = sorted(RAW_DIR.glob("*BUDDI_export*.csv"))
if not export_files:
    raise FileNotFoundError("No raw export CSV found matching '*BUDDI_export*.csv' in data/raw/")

if len(export_files) > 1:
    print(f"Multiple export files found. Using the first one: {export_files[0].name}")

export_path = export_files[0]

df_export_raw = load_csv_with_date_parsing(
    export_path,
    delimiter=DELIMITER,
    date_column_substring=DATE_COLUMN_SUBSTRING,
    dayfirst=DAYFIRST,
    date_errors="coerce",
)


df_peri_struct = raw_to_struct_perinatal(df_export_raw)
df_peri_sem = struct_to_sem_perinatal(df_peri_struct)

df_peri_struct.to_csv(OUT_STRUCT_DIR / STRUCT_OUT_NAME, index=False)
df_peri_sem.to_csv(OUT_SEM_DIR / SEM_OUT_NAME, index=False)

print("Perinatal STRUCT+SEM outputs written.")
print("Wrote:", OUT_STRUCT_DIR / STRUCT_OUT_NAME)
print("Wrote:", OUT_SEM_DIR / SEM_OUT_NAME)

Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_export_perinatal_history.csv
Perinatal STRUCT+SEM outputs written.
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_export_perinatal_history.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_export_perinatal_history.csv


## Neuropsychiatric History
Raw → structural → semantic


In [32]:
# -------- Neuropsychiatric history (STRUCT + SEM) --------
from jbhi_eval.io import load_csv_with_date_parsing
from jbhi_eval.transform_neuropsychiatric import (
    raw_to_struct_neuropsychiatric,
    struct_to_sem_neuropsychiatric,
    STRUCT_OUT_NAME,
    SEM_OUT_NAME,
)

# Load raw neuropsychiatric file
neuropsy_files = sorted(RAW_DIR.glob("*BUDDI_Neuropsychiatric*.csv"))
if not neuropsy_files:
    raise FileNotFoundError("No neuropsychiatric CSV found matching '*BUDDI_Neuropsychiatric*.csv' in data/raw/")

neuropsy_path = neuropsy_files[0]

df_neuropsy_raw = load_csv_with_date_parsing(
    neuropsy_path,
    delimiter=DELIMITER,
    date_column_substring=DATE_COLUMN_SUBSTRING,  # typically "date"
    dayfirst=DAYFIRST,
    date_errors="coerce",
)

df_neuropsy_struct = raw_to_struct_neuropsychiatric(df_neuropsy_raw)
df_neuropsy_sem = struct_to_sem_neuropsychiatric(df_neuropsy_struct)


df_neuropsy_struct.to_csv(OUT_STRUCT_DIR / STRUCT_OUT_NAME, index=False)
df_neuropsy_sem.to_csv(OUT_SEM_DIR / SEM_OUT_NAME, index=False)

neuropsy_raw_copy = copy_raw_to_processed(neuropsy_path, STRUCT_OUT_NAME.replace("struct_", "raw_", 1))
print("Raw copy ->", neuropsy_raw_copy)

print("Neuropsychiatric STRUCT+SEM outputs written.")
print("Wrote:", OUT_STRUCT_DIR / STRUCT_OUT_NAME)
print("Wrote:", OUT_SEM_DIR / SEM_OUT_NAME)

Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_export_neuropsychiatric_history.csv
Neuropsychiatric STRUCT+SEM outputs written.
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_export_neuropsychiatric_history.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_export_neuropsychiatric_history.csv


## RBS
Raw → structural → semantic


In [33]:
# -------- RBS (STRUCT + SEM) --------
from jbhi_eval.io import load_csv_with_date_parsing
from jbhi_eval.transform_rbs import (
    raw_to_struct_rbs,
    struct_to_sem_rbs,
    STRUCT_OUT_NAME,
    SEM_OUT_NAME,
)

rbs_files = sorted(RAW_DIR.glob("*BUDDI_RBS*.csv"))
if not rbs_files:
    raise FileNotFoundError("No RBS CSV found matching '*BUDDI_RBS*.csv' in data/raw/")

if len(rbs_files) > 1:
    print(f"Multiple RBS files found. Using the first one: {rbs_files[0].name}")

rbs_path = rbs_files[0]

df_rbs_raw = load_csv_with_date_parsing(
    rbs_path,
    delimiter=DELIMITER,
    date_column_substring=DATE_COLUMN_SUBSTRING,  
    dayfirst=DAYFIRST,
    date_errors="coerce",
)

df_rbs_struct = raw_to_struct_rbs(df_rbs_raw)
df_rbs_sem = struct_to_sem_rbs(df_rbs_struct)

df_rbs_struct.to_csv(OUT_STRUCT_DIR / STRUCT_OUT_NAME, index=False)
df_rbs_sem.to_csv(OUT_SEM_DIR / SEM_OUT_NAME, index=False)

rbs_raw_copy = copy_raw_to_processed(rbs_path, STRUCT_OUT_NAME.replace("struct_", "raw_", 1))
print("Raw copy ->", rbs_raw_copy)


print("RBS STRUCT+SEM outputs written.")
print("Wrote:", OUT_STRUCT_DIR / STRUCT_OUT_NAME)
print("Wrote:", OUT_SEM_DIR / SEM_OUT_NAME)

Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_RBS.csv
RBS STRUCT+SEM outputs written.
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_RBS.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_RBS.csv


## Lab Screening
Raw → structural → semantic


In [31]:
# -------- Screening lab (STRUCT + SEM) --------
from jbhi_eval.io import load_csv_with_date_parsing
from jbhi_eval.transform_lab import (
    raw_to_struct_lab,
    struct_to_sem_lab,
    STRUCT_OUT_NAME,
    SEM_OUT_NAME,
)

export_files = sorted(RAW_DIR.glob("*BUDDI_export_*.csv"))
if not export_files:
    raise FileNotFoundError("No raw export CSV found matching '*BUDDI_export_*.csv' in data/raw/")

if len(export_files) > 1:
    print(f"Multiple export files found. Using the first one: {export_files[0].name}")

export_path = export_files[0]

df_export_raw = load_csv_with_date_parsing(
    export_path,
    delimiter=DELIMITER,
    date_column_substring="lab_date",
    dayfirst=DAYFIRST,
    date_errors="coerce",
)

df_lab_struct = raw_to_struct_lab(df_export_raw)
df_lab_sem = struct_to_sem_lab(df_lab_struct)

df_lab_struct.to_csv(OUT_STRUCT_DIR / STRUCT_OUT_NAME, index=False)
df_lab_sem.to_csv(OUT_SEM_DIR / SEM_OUT_NAME, index=False)


print("Lab STRUCT+SEM outputs written.")
print("Wrote:", OUT_STRUCT_DIR / STRUCT_OUT_NAME)
print("Wrote:", OUT_SEM_DIR / SEM_OUT_NAME)

/Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/src/jbhi_eval/transform_lab.py:106: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)


Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_export_screening_lab.csv
Lab STRUCT+SEM outputs written.
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_export_screening_lab.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_export_screening_lab.csv


## SRS
Raw → structural → semantic


In [34]:
# -------- SRS (STRUCT + SEM) --------
from jbhi_eval.io import load_csv_with_date_parsing
from jbhi_eval.transform_srs import (
    raw_to_struct_srs,
    struct_to_sem_srs,
    STRUCT_OUT_NAME,
    SEM_OUT_NAME,
)

srs_files = sorted(RAW_DIR.glob("*BUDDI_SRS*.csv"))
if not srs_files:
    raise FileNotFoundError("No SRS CSV found matching '*BUDDI_SRS*.csv' in data/raw/")

if len(srs_files) > 1:
    print(f"Multiple SRS files found. Using the first one: {srs_files[0].name}")

srs_path = srs_files[0]


df_srs_raw = load_csv_with_date_parsing(
    srs_path,
    delimiter=DELIMITER,
    date_column_substring=DATE_COLUMN_SUBSTRING,  
    dayfirst=DAYFIRST,
    date_errors="coerce",
)


df_srs_struct = raw_to_struct_srs(df_srs_raw)
df_srs_sem = struct_to_sem_srs(df_srs_struct)

df_srs_struct.to_csv(OUT_STRUCT_DIR / STRUCT_OUT_NAME, index=False)
df_srs_sem.to_csv(OUT_SEM_DIR / SEM_OUT_NAME, index=False)

srs_raw_copy = copy_raw_to_processed(srs_path, STRUCT_OUT_NAME.replace("struct_", "raw_", 1))
print("Raw copy ->", srs_raw_copy)


print("SRS STRUCT+SEM outputs written.")
print("Wrote:", OUT_STRUCT_DIR / STRUCT_OUT_NAME)
print("Wrote:", OUT_SEM_DIR / SEM_OUT_NAME)

/Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/src/jbhi_eval/fhir_flatten.py:26: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)


Raw copy -> /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/raw/raw_BUDDI_SRS.csv
SRS STRUCT+SEM outputs written.
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/struct/struct_BUDDI_SRS.csv
Wrote: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem/sem_BUDDI_SRS.csv
